# Train football detector on Colab

Runs the same scripts as local development — nothing is reimplemented in this notebook, so
there is no second code path to keep in sync.

**Runtime → Change runtime type → GPU (T4 is enough for yolov8n @ 640).**

Pipeline: prepare data → train → export ONNX → **parity gate** → quantize → eval → benchmark.

## 0. Environment check

In [ ]:
!nvidia-smi
import sys; print(sys.version)

## 1. Get the repo

Set `REPO_URL` to your fork, or upload the folder to Drive and skip the clone.

In [ ]:
import os
from pathlib import Path

REPO_URL = ""  # e.g. https://github.com/<you>/football-detect-serve.git
WORKDIR = Path("/content/football-detect-serve")

if REPO_URL and not WORKDIR.exists():
    !git clone $REPO_URL {WORKDIR}

assert WORKDIR.exists(), f"{WORKDIR} missing — clone it or upload the folder"
os.chdir(WORKDIR)
print(Path.cwd())
!ls

In [ ]:
# Colab already ships torch; installing it again wastes several minutes.
!pip install -q ultralytics onnx onnxslim onnxruntime opencv-python-headless pyyaml

## 2. Dataset

Put the dataset zip URL in `configs/train.yaml` under `dataset.url`, or pass `--url` here.
`prepare_data.py` verifies the class map against `dataset.expected_classes` and **fails**
if the ids disagree — a silently permuted class map is the single most common cause of a
mAP that looks broken for no reason.

In [ ]:
# For a private Roboflow dataset:
#   from google.colab import userdata
#   os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

!python scripts/prepare_data.py

## 3. Train

All hyperparameters live in `configs/train.yaml`. Override per-run with `--set`.

In [ ]:
!python scripts/train.py --set epochs=100 batch=16 device=0

In [ ]:
from IPython.display import Image, display

for name in ["results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"]:
    p = Path("runs/detect/football") / name
    if p.exists():
        print(p)
        display(Image(str(p), width=900))

## 4. Export to ONNX (dynamic batch, simplified)

In [ ]:
!python scripts/export_onnx.py

## 5. GATE: parity

If this cell is non-zero, **stop** — the ONNX graph does not reproduce the torch model and
every downstream number is meaningless.

In [ ]:
rc = os.system("python scripts/check_parity.py")
assert rc == 0, "PARITY GATE FAILED — do not ship this export"
print("parity OK")

## 6. Baseline accuracy (fp32)

This becomes the reference the int8 model is gated against.

In [ ]:
!python scripts/eval_map.py --backend onnx --weights models/best.onnx

## 7. INT8 quantization + accuracy gate

In [ ]:
!cp reports/accuracy.json reports/accuracy.fp32_baseline.json
!python scripts/quantize_int8.py

In [ ]:
rc = os.system(
    "python scripts/eval_map.py --backend onnx --weights models/best.int8.onnx "
    "--compare-to reports/accuracy.fp32_baseline.json"
)
print("int8 accuracy gate:", "PASS" if rc == 0 else "FAIL")

## 8. Latency

Note: Colab's CPU is not your serving CPU. Treat these as relative numbers and re-run
`scripts/benchmark.py` on the target host before believing any absolute figure.

In [ ]:
!python scripts/benchmark.py --batch-sizes 1 2 4 8 --runs 30

In [ ]:
import json

rows = json.load(open("reports/latency.json"))["results"]
for r in sorted(rows, key=lambda r: (r["backend"], r["batch_size"])):
    print(f"{r['backend']:<12} bs={r['batch_size']:<3} p95={r['p95_ms']:7.2f}ms  {r['throughput_img_s']:7.1f} img/s")

## 9. Download the artifacts

In [ ]:
!zip -q -r artifacts.zip models/best.pt models/best.onnx models/best.int8.onnx reports/

from google.colab import files
files.download("artifacts.zip")